# 1. Dataset Intake & Provenance Audit

This notebook loads and validates the three data layers that support the historical replay analysis: the 12-case canonical dataset (expert-triangulated), the 91-case expanded benchmark, and the EEE triangulation overlay. It computes provenance statistics reported in the manuscript.

In [1]:
# Configuration
import json, csv, hashlib, statistics
from pathlib import Path

def _repo_root():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "config" / "harness_settings.json").is_file():
            return candidate
    return start

REPO_ROOT = _repo_root()
CANONICAL_PATH = REPO_ROOT / 'data' / 'canonical' / 'canonical_dataset.json'
BENCHMARK_DIR = REPO_ROOT / 'data' / 'benchmark' / 'cases'
EEE_PATH = REPO_ROOT / 'data' / 'benchmark' / 'eee_overlay' / 'canonicalised_v1.json'
CORE_EQUIV_PATH = REPO_ROOT / 'config' / 'core_equivalent_cases.json'
OUTPUT_DIR = REPO_ROOT / 'outputs' / 'tables'

## 1.1 Canonical Dataset (12 Expert-Triangulated Cases)

The primary analysis uses 12 historical AI governance failure cases, each encoded across 15 structural causal model (SCM) variables.

In [2]:
with open(CANONICAL_PATH) as f:
    canonical = json.load(f)

case_ids = sorted(canonical['cases'].keys())
n_cases = len(case_ids)
n_features = len(list(canonical['cases'].values())[0]['features'])
total_encodings = n_cases * n_features

print(f'Canonical dataset: {n_cases} cases x {n_features} features = {total_encodings} encodings')
for cid in case_ids:
    print(f'  {cid}')

assert n_cases == 12, f'Expected 12 cases, got {n_cases}'
assert n_features == 15, f'Expected 15 features, got {n_features}'
assert total_encodings == 180, f'Q21: Expected 180 encodings, got {total_encodings}'
print(f'\nQ21: {total_encodings} total encodings ✓')

Canonical dataset: 12 cases x 15 features = 180 encodings
  amazon_recruiting
  babylon
  compas
  epic_sepsis
  gender_shades
  google_dr
  google_flu
  ibm_watson
  microsoft_tay
  optum_health
  uber_av
  uk_alevels

Q21: 180 total encodings ✓


## 1.2 Provenance Classification

Each feature encoding carries a provenance classification indicating the strength of its evidential basis.

In [3]:
prov_counts = {}
confidence_values = []

for cid in case_ids:
    for fname, fobj in canonical['cases'][cid]['features'].items():
        pc = fobj.get('provenance_class', 'unknown')
        prov_counts[pc] = prov_counts.get(pc, 0) + 1
        confidence_values.append(fobj.get('confidence_level', 0.0))

direct = prov_counts.get('direct_evidence', 0)
rule = prov_counts.get('rule_derived', 0)
imputed = prov_counts.get('imputed_from_context', 0)
uncertain = prov_counts.get('uncertain_estimate', 0)

print('Provenance distribution:')
print(f'  direct_evidence:      {direct:3d} ({direct/180*100:.1f}%)')
print(f'  rule_derived:         {rule:3d} ({rule/180*100:.1f}%)')
print(f'  imputed_from_context: {imputed:3d} ({imputed/180*100:.1f}%)')
print(f'  uncertain_estimate:   {uncertain:3d} ({uncertain/180*100:.1f}%)')

assert direct == 49, f'Q22: Expected 49, got {direct}'
assert rule == 68, f'Q23: Expected 68, got {rule}'
assert imputed == 51, f'Q24: Expected 51, got {imputed}'
assert uncertain == 12, f'Q25: Expected 12, got {uncertain}'
print('\nQ22-Q25: Provenance counts ✓')

Provenance distribution:
  direct_evidence:       49 (27.2%)
  rule_derived:          68 (37.8%)
  imputed_from_context:  51 (28.3%)
  uncertain_estimate:    12 (6.7%)

Q22-Q25: Provenance counts ✓


## 1.3 Confidence Distribution

Feature-level confidence scores quantify encoding reliability. The mean confidence of 0.591 reflects the challenge of evidence-grounded governance assessment.

In [4]:
mean_conf = round(statistics.mean(confidence_values), 3)
median_conf = round(statistics.median(confidence_values), 3)
above_70 = sum(1 for c in confidence_values if c >= 0.70)

print(f'Mean confidence:   {mean_conf}')
print(f'Median confidence: {median_conf}')
print(f'Above 0.70:        {above_70}/180 ({above_70/180*100:.1f}%)')

assert mean_conf == 0.591, f'Q26: Expected 0.591, got {mean_conf}'
assert median_conf == 0.600, f'Q27: Expected 0.600, got {median_conf}'
assert above_70 == 71, f'Q28: Expected 71, got {above_70}'
print('\nQ26-Q28: Confidence metrics ✓')

Mean confidence:   0.591
Median confidence: 0.6
Above 0.70:        71/180 (39.4%)

Q26-Q28: Confidence metrics ✓


## 1.4 Expanded Benchmark Inventory

The expanded benchmark comprises 91 cases: 61 historical failures and 30 FDA-cleared control devices, producing 1,365 feature encodings.

In [5]:
benchmark_files = sorted(BENCHMARK_DIR.glob('*.json'))
n_benchmark = len(benchmark_files)
n_failures = sum(1 for f in benchmark_files if not f.name.startswith('control_'))
n_controls = sum(1 for f in benchmark_files if f.name.startswith('control_'))

with open(EEE_PATH) as f:
    eee = json.load(f)

print(f'Benchmark: {n_benchmark} cases ({n_failures} failures + {n_controls} controls)')
print(f'Total encodings: {n_benchmark * 15}')
print(f'EEE overlay: {eee["total_cases"]} cases ({eee["cases_processed"]} processed, {eee["cases_blocked"]} blocked)')

assert n_benchmark == 91
assert n_failures == 61
assert n_controls == 30
assert n_benchmark * 15 == 1365, f'Q36: Expected 1365'
print('\nQ36: 1,365 encodings ✓')

Benchmark: 91 cases (61 failures + 30 controls)
Total encodings: 1365
EEE overlay: 39 cases (37 processed, 2 blocked)

Q36: 1,365 encodings ✓


## 1.5 Core-Equivalent Cases

Eight additional cases were upgraded to methodological equivalence with the original 12 through EEE triangulation.

In [6]:
with open(CORE_EQUIV_PATH) as f:
    core_equiv = json.load(f)

core_ids = core_equiv['case_ids']
eee_lookup = {c['case_id']: c for c in eee['cases']}

total_overlay_feats = 0
print('Core-equivalent cases:')
for cid in core_ids:
    n_feats = len(eee_lookup[cid].get('eee_feature_overlay', {}))
    total_overlay_feats += n_feats
    print(f'  {cid}: {n_feats} overlay features')

assert len(core_ids) == 8
assert total_overlay_feats == 16, f'Expected 16 overlay features, got {total_overlay_feats}'
print(f'\nTotal overlay features: {total_overlay_feats} ✓')

Core-equivalent cases:
  covid_cxr_shortcuts_degrave2021: 2 overlay features
  sex_imbalance_thoracic_larrazabal2020: 2 overlay features
  hidden_stratification_oakden2020: 2 overlay features
  race_recognition_medical_images_gichoya2022: 2 overlay features
  dermatology_ml_disparities_adamson2018: 2 overlay features
  radiology_shortcut_bias_review_banerjee2023: 2 overlay features
  shortcutting_risk_imaging_hill2024: 2 overlay features
  adversarial_attacks_medical_ml_finlayson2019: 2 overlay features

Total overlay features: 16 ✓


## 1.6 Write Output

In [7]:
output_path = OUTPUT_DIR / 'dataset_inventory.csv'
with open(output_path, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['layer', 'source', 'cases', 'features_per_case', 'total_encodings'])
    w.writerow(['canonical', 'canonical_dataset.json', 12, 15, 180])
    w.writerow(['normalised', 'public_normalised_dataset.json', 12, 15, 180])
    w.writerow(['perturbation', 'perturbation_dataset.json', 12, 15, 180])
    w.writerow(['benchmark', 'cases/*.json', n_benchmark, 15, n_benchmark * 15])
    w.writerow(['eee_overlay', 'canonicalised_v1.json', eee['total_cases'], 'variable', eee['cases_processed']])
    w.writerow(['core_equivalent', 'core_equivalent_cases.json', len(core_ids), 'variable', total_overlay_feats])

print(f'Written: {output_path}')

Written: C:\Users\Walter.Brown\EAA_Portfolio\ethical-alpha-audit-paper-4-historical-replay\outputs\tables\dataset_inventory.csv
